### Work with the Survey Manager

In [1]:
import arcgis
from arcgis.gis import GIS
import datetime
from datetime import date, timedelta
import shutil
gis = GIS(username="NinjaGreen", password="Survey!23")

In [2]:
survey_manager = arcgis.apps.survey123.SurveyManager(gis)
surveys = survey_manager.surveys
assert len(surveys) > 0

NIIT%20Reverse%20Geocode
NIIT%20Pre%20Enumeration%20Survey
schools
Water%20Station%20Editing%20Example
form
Redlands%20Hydrant%20Inspection%202022
FDA%20Food%20Inspection%20Form%203-A
Water%20Leak
Political%20Canvassing%20v2
COVID-19%20Symptoms%20Check
Transmission%20Tower
PreEnumeration%20Survey
Mosquito%20Request
Inspect%20and%20Follow%20up
Spike%20Measurements
Annotation%20Custom%20Palette
Grid%20Theme%20Demo
form
form
Daily%20Fish%20Log
Spike
Street%20Service%20Report
Daily%20Airfield%20Safety%20Self-Inspection
%CE%A0%CE%99%CE%9B%CE%9F%CE%A4%CE%99%CE%9A%CE%97
Hidrantes%20v2
Mosquito_Master
form
Parking%20Meter%20Inspection%20(San%20Diego)
FDA%20Food%20Inspection%20Form%203-A%20cleanup
Mobile%20Estimating%20Gas%20Service%20Replacement%20v2
NIIT%20Daily%20Fish%20Log
SCAT%20Survey
form
Rapid%20Evaluation%20Safety%20Assessment%20Form
Campground%20Survey
Redlands%20Hydrant%20Inspection
Polio%20Monitoring
UC%202022%20Tree%20Inventory
form
Editing_Inbox_Demo
form
form
COVID-19%20Inpatient

In [3]:
survey_by_id = survey_manager.get("e7709174ba48426c880e504e11319970")
assert len(survey_by_id.properties['title']) > 0

Water%20Quality%20Inspection


In [4]:
forms = gis.content.search('type:form owner:NinjaGreen')
assert len(forms) > 0

In [5]:
survey_by_item = survey_manager.get(forms[8])
assert len(survey_by_item.properties) > 0

Political%20Canvassing%20v2


### Work with survey data

In [6]:
# Download - file formats
dl_formats = ['CSV', 'Shapefile', 'File Geodatabase']
for f in dl_formats:
    outfile = survey_by_id.download(f)
    assert outfile != None

In [7]:
# Download - pandas DataFrame
import pandas as pd
survey_df = survey_by_id.download('DF')
assert len(survey_df) > 0

,objectid,globalid,waterbodyname,waterbodytype,county,accesstype,ws_advisory,ws_advisory_desc,ws_advisory_start_date,ws_advisory_end_date,CreationDate,Creator,EditDate,Editor,ws_stationnumber,SHAPE
0,1,df92f362-08a9-4cec-8ebc-a4709f86adf5,Wolf Swamp,Swamp,None,Public Access,N/A,None,NaT,NaT,2018-07-12 01:47:21.840,NinjaGreen,2018-07-12 01:47:21.840000000,NinjaGreen,3980.0,"{""x"": -8804055.4873, ""y"": 4817689.304499999, ""..."
1,2,618cb3b6-5668-488c-bb77-2db0a92a8aca,New Germany Lake,Lake,Garrett,Public Access,No,None,NaT,NaT,2018-07-12 01:47:21.840,NinjaGreen,2018-07-12 01:47:21.840000000,NinjaGreen,5168.0,"{""x"": -8807705.8107, ""y"": 4813134.736199997, ""..."
2,3,32b39686-926e-4235-bfc5-382091ba74f0,Lake Louise,Lake,Garrett,Public Access,Yes,Algol Bloom,2018-07-12 21:42:00,2018-07-14 21:42:00,2018-07-12 01:47:21.840,NinjaGreen,2018-08-13 18:51:19.375000064,,5641.0,"{""x"": -8819832.7142, ""y"": 4820459.730700001, ""..."
3,4,d479a5e1-b65e-433d-acc6-e5baac583abf,Frostburg Reservoir,Reservoir,Garrett,Public Access,No,None,NaT,NaT,2018-07-12 01:47:21.840,NinjaGreen,2018-07-12 01:47:21.840000000,NinjaGreen,1612.0,"{""x"": -8795030.5798, ""y"": 4823277.554099999, ""..."
4,5,70e3119d-f94d-40ea-b827-7e09471b4559,"Herrington, Lake",Lake,Garrett,Public Access,No,None,NaT,NaT,2018-07-12 01:47:21.840,NinjaGreen,2018-07-12 01:47:21.840000000,NinjaGreen,2820.0,"{""x"": -8845043.3647, ""y"": 4787067.758699998, ""..."


### Create reports


In [8]:
# Identify report templates associated with a survey
templates = survey_by_id.report_templates
assert len(templates) > 0

In [9]:
# Generate a default report template 
temp_name = "Sample template " + str(datetime.datetime.now().strftime("%Y%m%d%H%M%S"))
new_template = survey_by_id.create_report_template(template_name=temp_name)
assert new_template != None

In [10]:
# Check template syntax
check = survey_by_id.check_template_syntax(new_template)
assert check['success'] == True

In [11]:
# Associate a report template with a survey
upload_template = survey_by_id.upload_report_template(template_file=new_template, template_name="PythonAPITemplate")
assert upload_template != None
templates = survey_by_id.report_templates
updated_templates = [x.title for x in templates]
assert 'PythonAPITemplate' in updated_templates

In [12]:
# Estimate credits
credits = survey_by_id.estimate(templates[0], where="1=1")
assert credits['success'] == True

In [13]:
# Create sample report
webmap = gis.content.search(query="title:Water Quality Inspection Web Map", item_type="Web Map")
wm_item=webmap[0]
sample = survey_by_id.create_sample_report(templates[0], where="objectid=1", utc_offset="-07:00", 
                                           report_title="Sample_Report", merge_files="none", survey_item=survey_by_id, 
                                           webmap_item=wm_item, map_scale="10000", locale='en')
assert sample != None

In [14]:
# Generate single report
report = survey_by_id.generate_report(templates[0], where="objectid=122")
assert report != None

In [15]:
# Generate multiple reports
local_batch_reports = survey_by_id.generate_report(templates[0], where="ws_advisory = 'Yes' and ws_advisory_start_date > '01/01/2020'", 
                                                   report_title="SingleReportInstance", package_name="ReportPackageNamePython")
assert local_batch_reports != None

In [16]:
# Generate multiple reports and save to your orgainzation
nowstring = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
folder_ID = survey_by_id.properties['ownerFolder']
org_batch_reports = survey_by_id.generate_report(templates[0], where="ws_advisory = 'Yes' and ws_advisory_start_date > '01/01/2020'",
                                                 package_name="Test_{0}".format(nowstring), folder_id=folder_ID)
assert org_batch_reports != None
search_for_batch_reports = gis.content.search("Test_{0}".format(nowstring))
assert len(search_for_batch_reports) > 0

In [17]:
# Generate report with all possible parameters
folder_ID = survey_by_id.properties['ownerFolder']
webmap = gis.content.search(query="title:Water Quality Inspection Web Map", item_type="Web Map")
wm_item=webmap[0]
all_params = survey_by_id.generate_report(templates[0], where="ws_advisory = 'Yes' and ws_advisory_start_date > '01/01/2020'", utc_offset="-07:00", report_title="All_Params_Report",
                                          package_name="All_Params_Package", output_format="pdf", folder_id=folder_ID, 
                                          merge_files="none", survey_item=survey_by_id, webmap_item=wm_item, 
                                          map_scale="10000", locale="en")
assert all_params != None

In [18]:
# Update report template
template_location = r"Sample_template.docx"
updated_template = shutil.copy(template_location, r'PythonAPITemplate.docx')
update = survey_by_id.update_report_template(updated_template)
assert len(update) > 0

In [19]:
recentReports = survey_by_id.reports
assert len(recentReports) > 0

In [20]:
assert upload_template.delete() == True
assert org_batch_reports.delete() == True
assert all_params.delete() == True

True